# Taller de práctica 4 · Proyecto reproducible y datos personales
**Programación y Análisis de Datos (MIACD02P01)** · Sábado 3 · Bloque 3 · ⏱ 40 min

**Objetivo:** organizar un análisis como proyecto reproducible (carpetas, funciones en .py, requirements, semillas, Git) y anonimizar datos personales según la LOPDP.

- Trabajo **individual** (Archivo → Guardar una copia en Drive). No es calificado: es práctica para el **Taller 2** y el **examen**.
- Cada ejercicio tiene una versión **base** (obligatoria) y algunos tienen **🚀 reto** (opcional).
- Completa las celdas marcadas con `# TU CÓDIGO AQUÍ`. Puedes usar IA, pero debes poder **explicar cada línea**.

In [36]:
import numpy as np
import pandas as pd

def generar_enemdu(n=4000, semilla=2026):
    """ENEMDU simplificada y SINTÉTICA (inspirada en la estructura del INEC).
    Los valores son simulados con relaciones realistas; no son datos oficiales."""
    rng = np.random.default_rng(semilla)
    prov = {"Pichincha": ("Sierra", .20), "Guayas": ("Costa", .24), "Azuay": ("Sierra", .06),
            "Manabí": ("Costa", .09), "El Oro": ("Costa", .05), "Tungurahua": ("Sierra", .04),
            "Imbabura": ("Sierra", .04), "Loja": ("Sierra", .04), "Los Ríos": ("Costa", .06),
            "Esmeraldas": ("Costa", .04), "Chimborazo": ("Sierra", .04), "Sucumbíos": ("Amazonía", .03),
            "Orellana": ("Amazonía", .03), "Morona Santiago": ("Amazonía", .03), "Galápagos": ("Insular", .01)}
    nombres = list(prov)
    p = np.array([prov[k][1] for k in nombres]); p = p / p.sum()
    provincia = rng.choice(nombres, n, p=p)
    region = np.array([prov[k][0] for k in provincia])
    area = np.where(rng.random(n) < np.where(region == "Amazonía", .45, .70), "Urbana", "Rural")
    sexo = rng.choice(["Hombre", "Mujer"], n, p=[.53, .47])
    edad = rng.integers(18, 66, n)
    anios_educ = np.clip(rng.normal(10 + 2.5 * (area == "Urbana"), 3.6, n), 0, 22).round()
    horas = np.clip(rng.normal(41, 10, n), 5, 80).round()
    ef_reg = pd.Series(region).map({"Sierra": .06, "Costa": 0, "Amazonía": -.10, "Insular": .35}).values
    log_ing = (5.05 + .075 * anios_educ + .011 * (edad - 18) - .00012 * (edad - 18) ** 2
               + .17 * (sexo == "Hombre") + .20 * (area == "Urbana") + ef_reg
               + .007 * (horas - 40) + rng.normal(0, .5, n))
    ingreso = np.exp(log_ing).round(2)
    adecuado = ((ingreso >= 470) & (horas >= 40)).astype(int)
    ruido = rng.random(n) < .05
    adecuado[ruido] = 1 - adecuado[ruido]
    df = pd.DataFrame({
        "id_persona": [f"P{i:05d}" for i in range(1, n + 1)],
        "provincia": provincia, "region": region, "area": area, "sexo": sexo,
        "edad": edad, "anios_educ": anios_educ, "horas_trabajo": horas,
        "ingreso": ingreso, "empleo_adecuado": adecuado})
    df["nivel_instruccion"] = pd.cut(df["anios_educ"], [-1, 6, 12, 16, 23],
                                     labels=["Primaria", "Secundaria", "Superior", "Posgrado"])
    # Problemas típicos de datos reales
    df.loc[rng.choice(n, int(n * .03), replace=False), "ingreso"] = np.nan
    df.loc[rng.choice(n, int(n * .02), replace=False), "anios_educ"] = np.nan
    atip = rng.choice(n, 12, replace=False)
    df.loc[atip, "ingreso"] = df.loc[atip, "ingreso"] * 15
    df = pd.concat([df, df.sample(25, random_state=semilla)], ignore_index=True)  # duplicados
    return df

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
SEMILLA = 42

df = generar_enemdu()
print(df.shape)
df.head()

(4025, 11)


,id_persona,provincia,region,area,sexo,edad,anios_educ,horas_trabajo,ingreso,empleo_adecuado,nivel_instruccion
0,P00001,Pichincha,Sierra,Rural,Hombre,52,6.00,29.00,213.81,0,Primaria
1,P00002,El Oro,Costa,Urbana,Hombre,55,18.00,56.00,665.82,1,Posgrado
2,P00003,Azuay,Sierra,Rural,Hombre,27,11.00,38.00,594.50,0,Secundaria
3,P00004,Guayas,Costa,Urbana,Mujer,63,10.00,46.00,"1,020.65",1,Secundaria
4,P00005,Guayas,Costa,Rural,Mujer,19,7.00,50.00,212.32,0,Secundaria


## Parte 1 · Estructura de proyecto
**Ejercicio 1.1:** crea las carpetas `mi_proyecto/data/crudos`, `data/procesados`, `notebooks`, `src` y `reportes`, y guarda el dataset crudo en `data/crudos/enemdu.csv`.

In [37]:
import os

carpetas = ["P4/data/crudos", "P4/data/procesados",
            "P4/notebooks", "P4/src", "P4/reportes"]

for carpeta in carpetas:
    os.makedirs(carpeta, exist_ok=True)   # exist_ok: no falla si ya existe

df.to_csv("P4/data/crudos/enemdu.csv", index=False)

# Verificamos la estructura creada
for raiz, dirs, archivos in os.walk("P4"):
    nivel = raiz.count(os.sep)
    print("    " * nivel + os.path.basename(raiz) + "/")
    for a in archivos:
        print("    " * (nivel + 1) + a)

P4/
    .gitignore
    README.md
    requirements.txt
    .git/
        COMMIT_EDITMSG
        config
        description
        HEAD
        index
        hooks/
            applypatch-msg.sample
            commit-msg.sample
            fsmonitor-watchman.sample
            post-update.sample
            pre-applypatch.sample
            pre-commit.sample
            pre-merge-commit.sample
            pre-push.sample
            pre-rebase.sample
            pre-receive.sample
            prepare-commit-msg.sample
            push-to-checkout.sample
            sendemail-validate.sample
            update.sample
        info/
            exclude
        logs/
            HEAD
            refs/
                heads/
                    main
        objects/
            0f/
                5001cea009c21f33e5ea9d3b01cc2b9afab87e
            20/
                a29e95e082a8983c0f076c1253ba7af87c0e5b
            32/
                793d44cb6e0b702847a6152823c577f2805499
            34/

## Parte 2 · Del notebook a funciones en un `.py`
**Ejercicio 2.1:** completa `src/limpieza.py` con dos funciones: `limpiar(df)` (quita duplicados, atípicos extremos y nulos) y `agregar_variables(df)` (crea `log_ingreso` y `grupo_edad`).

In [38]:
%%writefile P4/src/limpieza.py
"""Funciones de limpieza del proyecto."""
import numpy as np
import pandas as pd

def limpiar(df: pd.DataFrame) -> pd.DataFrame:    
    """Quita duplicados, atípicos extremos del ingreso (> Q3 + 3·IQR) y nulos."""
    df = df.drop_duplicates(subset="id_persona")
    q1, q3 = df["ingreso"].quantile([.25, .75])
    df = df[(df["ingreso"].isna()) | (df["ingreso"] <= q3 + 3 * (q3 - q1))]
    df = df.dropna(subset=["ingreso", "anios_educ"]).copy()
    return df

def agregar_variables(df: pd.DataFrame) -> pd.DataFrame:    
    """Crea log_ingreso y grupo_edad."""
    df = df.copy()
    df["log_ingreso"] = np.log(df["ingreso"])
    df["grupo_edad"] = pd.cut(df["edad"], bins=[17, 29, 44, 65],
                              labels=["18-29", "30-44", "45-65"])
    return df

Overwriting P4/src/limpieza.py


**Ejercicio 2.2:** importa tus funciones desde `src`, aplícalas al CSV crudo y guarda el resultado en `data/procesados/`.

In [39]:
import sys
sys.path.append("P4")
from src.limpieza import limpiar, agregar_variables

crudo = pd.read_csv("P4/data/crudos/enemdu.csv")
procesado = agregar_variables(limpiar(crudo))

procesado.to_csv("P4//data/procesados/enemdu_limpio.csv", index=False)
print("Crudo:", crudo.shape, "→ Procesado:", procesado.shape)
procesado.head()

Crudo: (4025, 11) → Procesado: (3744, 13)


,id_persona,provincia,region,area,sexo,edad,anios_educ,horas_trabajo,ingreso,empleo_adecuado,nivel_instruccion,log_ingreso,grupo_edad
0,P00001,Pichincha,Sierra,Rural,Hombre,52,6.00,29.00,213.81,0,Primaria,5.37,45-65
1,P00002,El Oro,Costa,Urbana,Hombre,55,18.00,56.00,665.82,1,Posgrado,6.50,45-65
2,P00003,Azuay,Sierra,Rural,Hombre,27,11.00,38.00,594.50,0,Secundaria,6.39,18-29
3,P00004,Guayas,Costa,Urbana,Mujer,63,10.00,46.00,"1,020.65",1,Secundaria,6.93,45-65
4,P00005,Guayas,Costa,Rural,Mujer,19,7.00,50.00,212.32,0,Secundaria,5.36,18-29


## Parte 3 · Semillas y `requirements.txt`
**Ejercicio 3.1:** ejecuta dos veces `df.sample(3)` **sin** semilla y dos veces **con** `random_state=SEMILLA`. ¿Qué cambia?

In [40]:
print("SIN semilla (1):", list(df.sample(3).index))
print("SIN semilla (2):", list(df.sample(3).index))
print("CON semilla (1):", list(df.sample(3, random_state=SEMILLA).index))
print("CON semilla (2):", list(df.sample(3, random_state=SEMILLA).index))

SIN semilla (1): [763, 405, 478]
SIN semilla (2): [1927, 3211, 893]
CON semilla (1): [1718, 1204, 2722]
CON semilla (2): [1718, 1204, 2722]


¿Qué cambia? Sin semilla, cada ejecución elige filas distintas, el resultado no se puede repetir. Con random_state=SEMILLA siempre salen las MISMAS filas, en cualquier computador: el análisis es reproducible (otra persona obtiene exactamente lo mismo).

**Ejercicio 3.2:** genera `requirements.txt` con las versiones **exactas** de las librerías que usa tu proyecto.

In [41]:
import sklearn, scipy, matplotlib
# TU CÓDIGO AQUÍ  (pista: cada librería tiene __version__)
import seaborn

librerias = {"numpy": np, "pandas": pd, "scikit-learn": sklearn,
             "scipy": scipy, "matplotlib": matplotlib, "seaborn": seaborn}

with open("P4/requirements.txt", "w") as f:
    for nombre, lib in librerias.items():
        f.write(f"{nombre}=={lib.__version__}\n")   # == fija la versión exacta

print(open("P4/requirements.txt").read())
# Para reinstalar el mismo entorno: pip install -r requirements.txt

numpy==2.5.3
pandas==3.0.4
scikit-learn==1.9.0
scipy==1.18.0
matplotlib==3.11.2
seaborn==0.13.2



## Parte 4 · README y Git
**Ejercicio 4.1:** escribe un `README.md` con: título, pregunta de análisis, fuente de datos, cómo ejecutar y estructura.

In [42]:
%%writefile p4/README.md
# Ingreso laboral y brecha de género en Ecuador (ENEMDU sintética)

## Pregunta de análisis
# ¿Qué factores se asocian con el ingreso laboral en Ecuador y la brecha entre hombres y mujeres se 
# mantiene al comparar personas con la misma educación, zona y región?

## Fuente de datos
# Encuesta ENEMDU **simplificada y sintética**, generada con la función `generar_enemdu()` 
# (semilla 2026) e inspirada en la estructura de la Encuesta Nacional de Empleo, Desempleo
# y Subempleo del INEC. Los valores son simulados: **no son datos oficiales**. 
# 4.000 personas y 11 variables (provincia, región, área, sexo, edad, años de educación, horas de trabajo, 
# ingreso, empleo adecuado y nivel de instrucción).

## Cómo ejecutar
1. Clonar el repositorio:
   `git clone https://github.com/USUARIO/REPO.git`
2. Instalar las dependencias:
   `pip install -r requirements.txt`
3. Colocar el archivo crudo en `data/crudos/enemdu.csv` (no se incluye en el repositorio).
4. Ejecutar los notebooks de la carpeta `notebooks/` en orden.

## Estructura
```
P4/
├── data/
│   ├── crudos/        # datos originales (excluidos de Git)
│   └── procesados/    # datos limpios generados por src/limpieza.py
├── notebooks/         # análisis exploratorio y modelos
├── src/
│   └── limpieza.py    # funciones limpiar(), agregar_variables(), anonimizar()
├── reportes/          # gráficos y resultados
├── requirements.txt   # versiones exactas de las librerías
└── README.md
```

Overwriting p4/README.md


**Ejercicio 4.2:** crea un `.gitignore` que excluya `data/crudos/` (puede tener datos personales) y los checkpoints de Jupyter. Luego inicializa Git y haz tu primer commit.

In [43]:
# TU CÓDIGO AQUÍ
# !cd p4 && git init && ...
gitignore = """# Datos crudos: pueden contener datos personales (LOPDP)
data/crudos/

# Checkpoints de Jupyter
.ipynb_checkpoints/

# Archivos de Python
__pycache__/
*.pyc

# Secretos (sal, claves)
.env
"""
with open("p4/.gitignore", "w") as f:
    f.write(gitignore)

# Git no guarda carpetas vacías: un archivo .gitkeep las mantiene en el repositorio
for carpeta in ["notebooks", "reportes"]:
    open(f"p4/{carpeta}/.gitkeep", "w").close()

!cd P4 && git init -b main
!git remote add origin https://github.com/Vidajame83/practica_4.git
#!cd P4 && git config user.name "Vidajame83" && git config user.email "victord.jaramillom@unilibre.edu.co"
!cd P4 && git add . && git commit -m "Estructura inicial del proyecto, limpieza, README y requirements"
!git push -u origin main
#!cd P4 && git status
#!cd P4 && git log --oneline

Reinitialized existing Git repository in C:/Users/VICTO/Documents/Maestría IA - Ciencia de Datos/1. Fund_Python_Ciencia_Datos/MIACD02P01/Sabado 3/Practica_4/P4/.git/


error: remote origin already exists.


On branch main
nothing to commit, working tree clean
branch 'main' set up to track 'origin/main'.


Everything up-to-date


## Parte 5 · Datos personales y LOPDP
Recibimos una versión con **datos personales ficticios** (cédula, nombre, fecha de nacimiento). Antes de compartirla debemos anonimizarla.

In [44]:
rng = np.random.default_rng(SEMILLA)
personal = df.drop_duplicates("id_persona").head(1500).copy()
personal["cedula"] = [f"{rng.integers(1, 25):02d}{rng.integers(0, 10**8):08d}" for _ in range(len(personal))]  # ficticias
personal["nombre"] = rng.choice(["Ana", "Luis", "María", "Jorge", "Carla", "Diego", "Sofía", "Pablo"], len(personal)) + " " + \
                     rng.choice(["Pérez", "Guerrero", "Andrade", "Zambrano", "Vera", "Mora", "Salazar"], len(personal))
personal["fecha_nac"] = pd.to_datetime("2026-01-01") - pd.to_timedelta(personal["edad"] * 365 + rng.integers(0, 365, len(personal)), unit="D")
personal[["cedula", "nombre", "fecha_nac", "provincia", "edad", "ingreso"]].head()

,cedula,nombre,fecha_nac,provincia,edad,ingreso
0,0377395604,María Pérez,1973-05-01,Pichincha,52,213.81
1,1643887843,Carla Vera,1970-04-10,El Oro,55,665.82
2,1185859791,Ana Mora,1998-01-27,Azuay,27,594.50
3,0369736802,Ana Pérez,1962-10-19,Guayas,63,"1,020.65"
4,0509417734,Pablo Zambrano,2006-12-04,Guayas,19,212.32


**Ejercicio 5.1:** anonimiza la tabla:
1. Elimina `nombre` y `fecha_nac` (identificadores directos).
2. Reemplaza `cedula` por un **seudónimo**: hash SHA-256 con una *sal* secreta, quedándote con los primeros 12 caracteres.
3. Generaliza `edad` en rangos de 10 años y redondea `ingreso` a la decena.

In [45]:
import hashlib
SAL = "cambia-esta-sal-secreta"   # nunca se sube a GitHub

def seudonimo(cedula, sal=SAL):
    """Hash SHA-256 de sal + cédula; se conservan los primeros 12 caracteres."""
    return hashlib.sha256((sal + str(cedula)).encode("utf-8")).hexdigest()[:12]

anon = personal.copy()

# 1. Eliminar identificadores directos
anon = anon.drop(columns=["nombre", "fecha_nac"])

# 2. Seudonimizar la cédula
anon["id_seudonimo"] = anon["cedula"].apply(seudonimo)
anon = anon.drop(columns=["cedula", "id_persona"])   # id_persona también enlaza con la base original

# 3. Generalizar edad (rangos de 10 años) y redondear ingreso a la decena
anon["rango_edad"] = pd.cut(anon["edad"], bins=[10, 20, 30, 40, 50, 60, 70], right=False,
                            labels=["10-19", "20-29", "30-39", "40-49", "50-59", "60-69"])
anon = anon.drop(columns=["edad"])
anon["ingreso"] = anon["ingreso"].round(-1)

print("Misma cédula → mismo seudónimo:", seudonimo("1712345678") == seudonimo("1712345678"))
anon[["id_seudonimo", "provincia", "sexo", "rango_edad", "ingreso"]].head()

Misma cédula → mismo seudónimo: True


,id_seudonimo,provincia,sexo,rango_edad,ingreso
0,ae9b47fb4df7,Pichincha,Hombre,50-59,210.00
1,ae9fc7994fa2,El Oro,Hombre,50-59,670.00
2,e5d5c1a94b79,Azuay,Hombre,20-29,590.00
3,d66cf359d0ee,Guayas,Mujer,60-69,"1,020.00"
4,c8aa81949387,Guayas,Mujer,10-19,210.00


**Ejercicio 5.2 · k-anonimato:** con los cuasi-identificadores (`provincia`, `rango_edad`, `sexo`), ¿cuál es el grupo más pequeño? Si hay grupos con menos de **k = 5** personas, alguien podría ser reidentificado.
Generaliza: provincias con menos de 100 personas → "Otras" y edad en 3 rangos amplios. Vuelve a medir.

In [46]:
# TU CÓDIGO AQUÍ
cuasi = ["provincia", "rango_edad", "sexo"]

def medir_k(data, columnas):
    grupos = data.groupby(columnas, observed=True).size()
    print(f"Grupos: {len(grupos)} | grupo más pequeño (k) = {grupos.min()} | "
          f"grupos con menos de 5 personas: {(grupos < 5).sum()}")
    return grupos

print("ANTES de generalizar:")
grupos_antes = medir_k(anon, cuasi)
print(grupos_antes.sort_values().head(), "\n")

# Generalización
anon_k = anon.copy()
conteo_prov = anon_k["provincia"].value_counts()
pequenas = conteo_prov[conteo_prov < 100].index
anon_k["provincia"] = anon_k["provincia"].where(~anon_k["provincia"].isin(pequenas), "Otras")

anon_k["rango_edad"] = pd.cut(personal.loc[anon_k.index, "edad"], bins=[17, 34, 49, 65],
                              labels=["18-34", "35-49", "50-65"])

print("DESPUÉS de generalizar:")
grupos_despues = medir_k(anon_k, cuasi)
print(grupos_despues.sort_values().head())

ANTES de generalizar:
Grupos: 170 | grupo más pequeño (k) = 1 | grupos con menos de 5 personas: 57
provincia   rango_edad  sexo  
Azuay       10-19       Mujer     1
Chimborazo  10-19       Hombre    1
El Oro      10-19       Hombre    1
Chimborazo  60-69       Mujer     1
Galápagos   30-39       Hombre    1
dtype: int64 

DESPUÉS de generalizar:
Grupos: 24 | grupo más pequeño (k) = 14 | grupos con menos de 5 personas: 0
provincia  rango_edad  sexo  
Manabí     50-65       Hombre    14
           35-49       Hombre    19
                       Mujer     20
           18-34       Mujer     23
                       Hombre    24
dtype: int64


Antes había combinaciones con 1-4 personas: alguien que conozca la provincia, la edad y el sexo de una persona podría reidentificarla. Tras generalizar, cada combinación tiene al menos 5 personas (k ≥ 5): se pierde detalle, pero se gana privacidad.

✍️ **Reflexión (conecta con tu proyecto de visión por computador):** el contador de personas y el lector de placas procesan **datos personales** (rostros, placas). Según la LOPDP, ¿qué deberías hacer antes de publicar tus videos o resultados? Menciona 2 medidas.

Los rostros y las placas vehiculares son **datos personales** según la LOPDP, porque permiten identificar a una persona directa o indirectamente (el rostro además puede ser un dato biométrico, que es sensible). Antes de publicar videos o resultados aplicaría:

1. **Anonimizar las imágenes:** difuminar (*blur*) o pixelar automáticamente los rostros y las placas en los videos antes de publicarlos, y en los resultados guardar solo lo necesario (por ejemplo, el **conteo** de personas o una placa **seudonimizada con hash y sal**, nunca la imagen o el texto original). Esto aplica los principios de **minimización** y **seudonimización**.
2. **Base legal y transparencia:** grabar solo con **consentimiento** o con una base legítima, informar con avisos visibles que se está grabando y para qué fin, y usar los datos **solo para ese propósito** (limitación de la finalidad).

Además: guardar los videos originales en un lugar seguro y **fuera de GitHub** (en `.gitignore`), con acceso restringido, y **eliminarlos** cuando ya no se necesiten (plazo de conservación).

### 🚀 Reto
Agrega a `src/limpieza.py` una función `anonimizar(df, sal)` con lo del ejercicio 5.1, haz un segundo commit y muestra `git log --oneline`.

In [47]:
# El texto exacto de la función del reto
reto_anonimizar = '''

def anonimizar(df: pd.DataFrame, sal: str) -> pd.DataFrame:
    """Elimina identificadores directos, seudonimiza la cedula y generaliza edad e ingreso."""
    import hashlib
    df = df.drop(columns=["nombre", "fecha_nac"]).copy()
    df["id_seudonimo"] = df["cedula"].apply(
        lambda c: hashlib.sha256((sal + str(c)).encode("utf-8")).hexdigest()[:12])
    df = df.drop(columns=["cedula", "id_persona"])
    df["rango_edad"] = pd.cut(df["edad"], bins=[10, 20, 30, 40, 50, 60, 70], right=False,
                             labels=["10-19", "20-29", "30-39", "40-49", "50-59", "60-69"])
    df = df.drop(columns=["edad"])
    df["ingreso"] = df["ingreso"].round(-1)
    return df
'''

# "a" de append (agregar al final) especificando codificación utf-8
with open("P4/src/limpieza.py", "a", encoding="utf-8") as f:
    f.write(reto_anonimizar)

print("¡Función del reto agregada exitosamente al final de limpieza.py!")

¡Función del reto agregada exitosamente al final de limpieza.py!


In [48]:
import importlib
import src.limpieza
importlib.reload(src.limpieza)
from src.limpieza import anonimizar

In [51]:
!cd P4
!git add .
!git commit -m "Reto completado: se agrega funcion anonimizar a src/limpieza.py"
!git log --oneline
!git push

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
50851dd Reto completado: se agrega funcion anonimizar a src/limpieza.py
8e70e68 Corregir estructura de carpeta P4
0770309 Estructura correcta de Practica 4


To https://github.com/Vidajame83/practica_4.git
   8e70e68..50851dd  main -> main
